# aqui rodamos 14 meses Mistral-7B-Instruct-v0.3-AWQ




In [2]:
PROMPT_TEXT = r"""<|im_start|>system
Return ONLY valid JSON: {{"drop":[...]}}. No extra text.

Title: {job_ad_title}
Sector: {job_sector_category}
Domain: {domain}
Tasks: {tasks_str}

Candidates (1-based):
{numbered}

Rules:
- Keep 2–3 by default.
- Keep 1 only if very sure all others are clearly wrong.
- Drop only clearly wrong functions.

Output ONLY JSON: {{"drop":[...]}}
""".strip()


## bge e5 and bge all in the same batch

In [1]:
import subprocess, re
from pathlib import Path

#SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/run_deepseek_drop_bge.sbatch")
SBATCH = Path("/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/sbatch/z_vllm_mistral_local_drop_all_embeds.sbatch")
assert SBATCH.exists(), f"Missing sbatch file: {SBATCH}"

out = subprocess.check_output(["sbatch", str(SBATCH)], text=True)
jobid = re.search(r"(\d+)", out).group(1)
print("JOBID:", jobid)
bge_jobid = jobid

JOBID: 2206726


# monitoring

In [2]:
import time, subprocess
from pathlib import Path

LOG_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/logs")
def sh(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout.strip()
while True:
    print("\n=== squeue ===")
    sq = sh(f"squeue -j {jobid}")
    print(sq if sq else "<finished>")

    out_logs = sorted(LOG_DIR.glob(f"*{jobid}*.out"))
    err_logs = sorted(LOG_DIR.glob(f"*{jobid}*.err"))

    if out_logs:
        print("\n--- STDOUT ---")
        print(sh(f"tail -n 20 {out_logs[-1]}"))

    if err_logs:
        print("\n--- STDERR ---")
        print(sh(f"tail -n 20 {err_logs[-1]}"))

    if not sq:
        break

    time.sleep(5)



=== squeue ===
<finished>

--- STDOUT ---
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=269443) INFO 02-08 13:52:05 [parallel_state.py:1208] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=269443) WARNING 02-08 13:52:05 [topk_topp_sampler.py:66] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
(EngineCore_DP0 pid=269443) INFO 02-08 13:52:05 [gpu_model_runner.py:2602] Starting to load model /projects/a5u/adu_dev/aisi-economy-index/models/mistral_awq...
(EngineCore_DP0 pid=269443) INFO 02-08 13:52:06 [gpu_model_runner.py:2634] Loading model from scratch...
(EngineCore_DP0 pid=269443) INFO 02-08 13:52:06 [cuda.py:366] Using Flash Attention backend on V1 engine.
(EngineCore_D

In [3]:
import os
import json
from pathlib import Path

# Base directory for your production runs
BASE_DIR = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/mistral")

# Models to check
MODELS = ["bge_large", "e5_large", "gte_large"]

def print_most_recent_outputs():
    print(f"{'MODEL':<10} | {'LATEST FILENAME'}")
    print("-" * 60)
    
    for model_dir_name in MODELS:
        # Construct path to the month folder
        target_path = BASE_DIR / model_dir_name / "adzuna_month01"
        
        if not target_path.exists():
            print(f"{model_dir_name:<10} | Path not found: {target_path}")
            continue
            
        # Get all jsonl files in the directory
        files = list(target_path.glob("*.jsonl"))
        
        if not files:
            print(f"{model_dir_name:<10} | No .jsonl files found.")
            continue
            
        # Find the most recent file based on modification time (mtime)
        latest_file = max(files, key=os.path.getmtime)
        
        print(f"{model_dir_name:<10} | {latest_file.name}")
        
        # Optional: Print the last 2 records from this specific latest file
        try:
            with open(latest_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
                if lines:
                    last_record = json.loads(lines[-1])
                    print(f"   -> Last Job ID: {last_record.get('job_id')}")
                    print(f"   -> Kept: {last_record.get('final')}")
        except Exception as e:
            print(f"   -> Error reading file: {e}")
        print("-" * 60)

if __name__ == "__main__":
    print_most_recent_outputs()

MODEL      | LATEST FILENAME
------------------------------------------------------------
bge_large  | vllm_drop_adzuna_month01_0_1000000000_job2206727_task0_rank0_20260208_135119.jsonl
   -> Last Job ID: 2767917838
   -> Kept: ['First-Line Supervisors of Retail Sales Workers', 'Stockers and Order Fillers', 'Order Clerks']
------------------------------------------------------------
e5_large   | vllm_drop_adzuna_month01_0_1000000000_job2206741_task14_rank0_20260208_135123.jsonl
   -> Last Job ID: 2767917838
   -> Kept: ['Sales Managers', 'First-Line Supervisors of Retail Sales Workers']
------------------------------------------------------------
gte_large  | vllm_drop_adzuna_month01_0_1000000000_job2206755_task28_rank0_20260208_135123.jsonl
   -> Last Job ID: 2767917838
   -> Kept: ['Stockers and Order Fillers', 'First-Line Supervisors of Non-Retail Sales Workers', 'Transportation, Storage, and Distribution Managers']
------------------------------------------------------------


## Report deepseek

In [3]:
import os
os.getcwd()
import importlib
import gen_report_helper as grh # gen_report_helper.py 
from pathlib import Path
importlib.reload(grh) 

res = grh.generate_LLM_MODEL_full_report_and_plots(
    JSONL_BASE_DIR=Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/mistral"),
    NPZ_BASE_DIR=Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection"),
    llm_model="Mistral-7B-Instruct-v0.3-AWQ",
    prompt_text=PROMPT_TEXT,
    reports_subdir="reports/mistral"
)
res


Scanning Month 1 to 14 (Flexible format) for embeds: ('bge_large', 'e5_large', 'gte_large')
[SELECT] bge_large | adzuna_month01 -> vllm_drop_adzuna_month01_0_1000000000_job2206727_task0_rank0_20260208_135119.jsonl
[SELECT] bge_large | adzuna_month02 -> vllm_drop_adzuna_month02_0_1000000000_job2206728_task1_rank0_20260208_135119.jsonl
[SELECT] bge_large | adzuna_month03 -> vllm_drop_adzuna_month03_0_1000000000_job2206729_task2_rank0_20260208_135119.jsonl
[SELECT] bge_large | adzuna_month04 -> vllm_drop_adzuna_month04_0_1000000000_job2206730_task3_rank0_20260208_135119.jsonl
[SELECT] bge_large | adzuna_month05 -> vllm_drop_adzuna_month05_0_1000000000_job2206731_task4_rank0_20260208_135121.jsonl
[SELECT] bge_large | adzuna_month06 -> vllm_drop_adzuna_month06_0_1000000000_job2206732_task5_rank0_20260208_135121.jsonl
[SELECT] bge_large | adzuna_month07 -> vllm_drop_adzuna_month07_0_1000000000_job2206733_task6_rank0_20260208_135119.jsonl
[SELECT] bge_large | adzuna_month08 -> vllm_drop_adzu

{'out_dir': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/mistral/20260210_222228',
 'txt_path': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/mistral/20260210_222228/20260210_222228_FULL_TEXT_REPORT.txt',
 'heatmap_path': '/lus/lfs1aip2/projects/a5u/adu_dev/aisi-economy-index/nbs/AISI_demo/Stage_3_negative_selection/dev/reports/mistral/20260210_222228/20260210_222228_avg_jaccard_heatmap.png'}